# Step 1: Clone

In [1]:
from __future__ import annotations
import csv, os, re, subprocess, time
from pathlib import Path
from typing import Optional, List

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT    = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
URL_LIST_CSV = WORK_ROOT / "URL_List.csv"
CLONE_ROOT   = WORK_ROOT / "clonesV8.1"
MANIFEST_CSV = WORK_ROOT / "clones_manifestV8.1.csv"

WITH_SUBMODULES = False     # set True if you want submodules initialized/updated
WITH_LFS        = False     # set True if you need Git LFS objects
FETCH_PR_REFS   = True      # set False to skip GitHub PR heads

# Create dirs
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helpers
# -----------------------------
DEFAULT_TIMEOUT = 1800  # 30 minutes for huge repos
GIT_ENV = {"GIT_TERMINAL_PROMPT": "0", "GIT_ASKPASS": ""}

def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True,
       capture: bool = True, timeout: Optional[int] = DEFAULT_TIMEOUT) -> subprocess.CompletedProcess:
    env = os.environ.copy()
    env.update(GIT_ENV)
    return subprocess.run(cmd, cwd=cwd, check=check,
                          capture_output=capture, text=True,
                          timeout=timeout, env=env)

def sh_ok(cmd: List[str], cwd: Optional[Path] = None, timeout: Optional[int] = DEFAULT_TIMEOUT) -> str:
    cp = sh(cmd, cwd=cwd, check=True, capture=True, timeout=timeout)
    return cp.stdout

def repo_dir_name_from_url(url: str) -> str:
    u = url.strip()
    # SSH form: git@host:owner/repo(.git)
    m = re.match(r"^[^@]+@([^:]+):([^/]+)/(.+?)(?:\.git)?$", u)
    if m:
        owner, name = m.group(2), m.group(3)
        return f"{owner}__{name}"
    # https://host/owner/repo(.git)
    if "://" in u:
        base = u.split("://", 1)[1]
    else:
        base = u
    parts = [p for p in base.split("/") if p]
    if len(parts) >= 2:
        owner, name = parts[-2], parts[-1].removesuffix(".git")
        return f"{owner}__{name}"
    return base.replace("/", "__").removesuffix(".git")

def ensure_full_clone(url: str, dest_root: Path,
                      with_submodules: bool = False,
                      with_lfs: bool = False,
                      fetch_pr_refs: bool = True) -> Path:
    """
    Ensures a non-shallow clone with full history of all branches & tags.
    Optionally fetches GitHub PR heads to refs/remotes/origin/pr/*,
    initializes submodules, and fetches LFS objects.
    """
    dest_root.mkdir(parents=True, exist_ok=True)
    d = dest_root / repo_dir_name_from_url(url)

    if not (d.exists() and (d / ".git").exists()):
        # no-recurse-submodules avoids submodule cost unless requested later
        sh(["git", "clone", "--no-recurse-submodules", "--tags", url, str(d)],
           capture=False)
    else:
        # If repo exists, ensure origin URL is correct
        sh(["git", "remote", "set-url", "origin", url], cwd=d, check=False)

    # If shallow, unshallow
    cp = sh(["git", "rev-parse", "--is-shallow-repository"], cwd=d, check=False)
    if cp.returncode == 0 and cp.stdout.strip() == "true":
        sh(["git", "fetch", "--unshallow", "--tags"], cwd=d, capture=False)
    else:
        # Make sure we have tags even if already full
        sh(["git", "fetch", "--tags"], cwd=d, check=False, capture=False)

    # Fetch all branches under refs/heads/* and prune deleted ones
    sh(["git", "fetch", "origin", "--prune", "--tags",
        "+refs/heads/*:refs/remotes/origin/*", "--quiet"], cwd=d, check=False, capture=False)

    # Best-effort PR refs (GitHub); harmless if not present
    if fetch_pr_refs:
        sh(["git", "fetch", "origin",
            "+refs/pull/*/head:refs/remotes/origin/pr/*", "--quiet"], cwd=d, check=False, capture=False)

    # Optional: submodules
    if with_submodules:
        sh(["git", "submodule", "update", "--init", "--recursive"], cwd=d, capture=False)

    # Optional: LFS
    if with_lfs:
        # If git-lfs isn't installed, these will fail harmlessly due to check=False
        sh(["git", "lfs", "install"], cwd=d, check=False, capture=False)
        sh(["git", "lfs", "fetch", "--all"], cwd=d, check=False, capture=False)
        sh(["git", "lfs", "checkout"], cwd=d, check=False, capture=False)

    return d

def get_total_commits(repo_dir: Path) -> int:
    # Count across all refs reachable in the repository
    cp = sh(["git", "rev-list", "--all", "--count"], cwd=repo_dir, check=False)
    try:
        return int((cp.stdout or "0").strip() or "0")
    except Exception:
        return 0

# -----------------------------
# Main: FULL CLONE ONLY
# -----------------------------
assert URL_LIST_CSV.exists(), f"CSV not found: {URL_LIST_CSV}"

rows, ok, fail = [], 0, 0
with URL_LIST_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        url = (row.get("repo_url") or "").strip()
        if not url:
            continue
        t0 = time.time()
        rec = {
            "repo_url": url,
            "dir": None,
            "status": "unknown",
            "seconds": None,
            "total_commits": None,
            "error": ""
        }
        try:
            d = ensure_full_clone(url, CLONE_ROOT,
                                  with_submodules=WITH_SUBMODULES,
                                  with_lfs=WITH_LFS,
                                  fetch_pr_refs=FETCH_PR_REFS)
            rec["dir"] = str(d)
            rec["total_commits"] = get_total_commits(d)
            rec["status"] = "ok"
            ok += 1
        except subprocess.CalledProcessError as e:
            rec["status"] = "error"
            rec["error"]  = (e.stderr or e.stdout or str(e)).strip()[:2000]
            fail += 1
        except Exception as e:
            rec["status"] = "error"
            rec["error"]  = str(e)[:2000]
            fail += 1

        rec["seconds"] = round(time.time() - t0, 2)
        rows.append(rec)
        print(f"[{rec['status']}] {url} -> {rec['dir']} ({rec['seconds']}s)  commits={rec['total_commits']}")

# Write manifest
MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
fieldnames = ["repo_url","dir","status","seconds","total_commits","error"]
with MANIFEST_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows)

print(f"\nDone. OK={ok}, FAIL={fail}. Manifest: {MANIFEST_CSV}")


[ok] https://github.com/Rajawali/Rajawali -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV8.1\Rajawali__Rajawali (19.6s)  commits=3065
[ok] https://github.com/splitwise/TokenAutoComplete -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV8.1\splitwise__TokenAutoComplete (1.65s)  commits=428
[ok] https://github.com/OpnTec/bodyapps-android -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV8.1\OpnTec__bodyapps-android (2.02s)  commits=160
[ok] https://github.com/Stuart-campbell/RushOrm -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV8.1\Stuart-campbell__RushOrm (1.48s)  commits=216
[ok] https://github.com/kost/NetworkMapper -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV8.1\kost__NetworkMapper (1.09s)  commits=71
[ok] https://github.com/Redgram/redgram-for-reddit -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV8.1\Redgram__redgram-for-reddit (1.95s)  commits=328
[ok] https://githu

In [ ]:
# RQ2 — Step 2 (v8.1+audit, patched FINAL): Mine commit snapshots (normalized + helpers)
# Patches retained from your version:
# - FIX: STEP_START_RX matches any step header (- name:/- id:/- uses:/- run:/etc.)
# - FIX: Emulator 'uses:' detection allows quoted values ('"reactivecircus/..."')
# - FIX: HEAD audit is more tolerant (looks for runner action strings even without 'uses:')
#
# NEW (minimal, backward-compatible) to fix the 9 repos without CCE:
# - Append file CONTENT to each emitted row (plus optional language + size). Nothing else removed.
# - File name pattern remains <repo>.jsonl so you can merge with existing mined outputs.
# - No changes to commit filtering, detectors, features, or audit CSV.

from __future__ import annotations
import os, re, json, subprocess, datetime as dt
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any

try:
    from zoneinfo import ZoneInfo  # Python 3.9+
except Exception:
    ZoneInfo = None

# ----------------------------- Config -----------------------------
WORK_ROOT      = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
CLONE_ROOT     = WORK_ROOT / "clonesV8.1"          # Can point this to a subset folder if you want to run on 9 repos
SNAPSHOT_DIR   = WORK_ROOT / "snapshotsV8.1"
AUDIT_CSV      = WORK_ROOT / "snapshots_auditV8.1.csv"
MAX_COMMITS_PER_REPO = 0  # 0 = no limit
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

# Toggle to include optional helper fields (safe to keep True)
INCLUDE_LANGUAGE = True
INCLUDE_SIZE     = True

CUTOFF_TZ_NAME = "America/Toronto"
_CUTOFF_DATE   = (2025, 8, 10, 23, 59, 59)
if ZoneInfo is not None:
    _tz = ZoneInfo(CUTOFF_TZ_NAME)
    _local_dt = dt.datetime(*_CUTOFF_DATE, tzinfo=_tz)
    CUTOFF_EPOCH = int(_local_dt.timestamp())
    CUTOFF_BEFORE_STR = _local_dt.strftime("%Y-%m-%d %H:%M:%S %z")
else:
    _utc_dt = dt.datetime(2025, 8, 11, 3, 59, 59, tzinfo=dt.timezone.utc)
    CUTOFF_EPOCH = int(_utc_dt.timestamp())
    CUTOFF_BEFORE_STR = "2025-08-10 23:59:59 -0400"

print(f"[cutoff] Using commit cutoff <= {CUTOFF_BEFORE_STR} (epoch={CUTOFF_EPOCH})")

def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    env = os.environ.copy()
    env["GIT_PAGER"] = ""
    return subprocess.run(
        cmd, cwd=str(cwd) if cwd else None, check=check,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        text=True, encoding="utf-8", errors="replace", env=env,
    )

# ---------------- CI file detection ----------------
CI_VENDOR_PATTERNS = [
    re.compile(r'(?i)(?:^|/)\.travis\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.appveyor\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)appveyor\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)circle\.yml$'),
    re.compile(r'(?i)(?:^|/)\.circleci/config\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)azure-pipelines\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.github/workflows/.*\.(yml|yaml)$'),
    re.compile(r'(?i)(?:^|/)bitbucket-pipelines\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.gitlab-ci\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.gitlab-ci\.yaml$'),
    re.compile(r'(?i)(?:^|/)Jenkinsfile(?:\.\w+)?$'),
    re.compile(r'(?i)(?:^|/)bitrise\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)bamboo\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)codeship-services\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.gocd\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.cirrus\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)wercker\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.semaphore\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)codemagic\.ya?ml$'),
]
GENERIC_CI_DIRS = re.compile(r'(?i)(?:^|/)(?:ci|\.ci|\.jenkins)(?:/|$)')

def is_ci_file(p: str) -> bool:
    if not p: return False
    if any(rx.search(p) for rx in CI_VENDOR_PATTERNS): return True
    return bool(GENERIC_CI_DIRS.search(p))

# ---------------- file types ----------------
GRADLE_FILES = [
    "build.gradle","build.gradle.kts","settings.gradle","settings.gradle.kts",
    "gradle.properties","gradle/wrapper/gradle-wrapper.properties",
]
SCRIPT_EXTS = {".sh",".bash",".zsh",".py",".bat",".cmd",".ps1",".psm1"}

# ADDED: lightweight language inference (optional)
EXT_TO_LANG = {
    ".java": "java",
    ".kt": "kotlin",
    ".kts": "kotlin",
    ".gradle": "gradle",
    ".xml": "xml",
    ".yml": "yaml",
    ".yaml": "yaml",
    ".properties": "properties",
    ".pro": "proguard",
    ".sh": "bash",
    ".py": "python",
    ".bat": "batch",
    ".cmd": "batch",
}

def infer_lang(path: str) -> Optional[str]:
    lp = path.lower()
    for ext, lang in EXT_TO_LANG.items():
        if lp.endswith(ext):
            return lang
    if is_gradle_file(path):
        return "gradle"
    return None

def is_gradle_file(p: str) -> bool:
    lp = p.lower()
    return any(lp.endswith(x) for x in (f.lower() for f in GRADLE_FILES))

def is_script_file(p: str) -> bool:
    lp = p.lower()
    return any(lp.endswith(ext) for ext in SCRIPT_EXTS)

def touched_relevant(paths: List[str]) -> bool:
    for p in paths:
        if not p.strip(): continue
        if is_ci_file(p) or is_gradle_file(p) or is_script_file(p):
            return True
    return False

# ---------------- git helpers ----------------
def list_relevant_commits(repo_dir: Path) -> List[Tuple[str,int,List[str]]]:
    cp = sh([
        "git", "-c", "i18n.logOutputEncoding=UTF-8", "-c", "core.quotepath=off",
        "log", "--all", "--before", CUTOFF_BEFORE_STR,
        "--name-only", "--pretty=%H%x09%ct"
    ], cwd=repo_dir)
    results: List[Tuple[str,int,List[str]]] = []
    sha: Optional[str] = None; ts: Optional[int] = None; changed: List[str] = []
    for line in cp.stdout.splitlines():
        if re.match(r"^[0-9a-f]{40}\t\d+$", line):
            if sha and ts and ts <= CUTOFF_EPOCH and touched_relevant(changed):
                results.append((sha, ts, changed))
            sha, ts_s = line.split("\t", 1); ts = int(ts_s); changed = []
        else:
            if line.strip(): changed.append(line.strip())
    if sha and ts and ts <= CUTOFF_EPOCH and touched_relevant(changed):
        results.append((sha, ts, changed))
    results.reverse()
    return results

def git_show(repo_dir: Path, sha: str, path: str) -> Optional[str]:
    try:
        cp = sh(["git", "show", f"{sha}:{path}"], cwd=repo_dir)
        return cp.stdout
    except subprocess.CalledProcessError:
        return None

def git_subject(repo_dir: Path, sha: str) -> str:
    try:
        cp = sh(["git","-c","i18n.logOutputEncoding=UTF-8","show","-s","--format=%s", sha],
                cwd=repo_dir, check=False)
        return (cp.stdout or "").strip()
    except Exception:
        return ""

RE_INT = re.compile(r"\d+")

# ---------------- Invocation detectors (DIY/GMD/Generic) ----------------
DIY_RX = re.compile(
    r"(?:\bemulator(?:\.bat|\.exe)?\s-|"
    r"\bavdmanager\b|"
    r"\bsdkmanager\b|"
    r"\bcreate\s+avd\b|"
    r"\badb\s+(?:-s\s+\S+\s+)?wait-for-device\b|"
    r"\badb\s+shell\s+getprop\s+sys\.boot_completed\b|"
    r"\bqemu\b)",
    re.I,
)

# FIX: allow optional quotes around uses: values
REACTIVECIRCUS_RX = re.compile(r"uses:\s*['\"]?reactivecircus/android-emulator-runner@[\w.\-]+['\"]?", re.I)
MALINSKIY_RX      = re.compile(r"uses:\s*['\"]?malinskiy/action-android/emulator-run-cmd@[\w.\-]+['\"]?", re.I)
EMULATOR_RUNNER_USES_RX = re.compile(
    r"(?:uses:\s*['\"]?reactivecircus/android-emulator-runner@[\w.\-]+['\"]?"
    r"|uses:\s*['\"]?malinskiy/action-android/emulator-run-cmd@[\w.\-]+['\"]?)",
    re.I
)

# ---------------- YAML step/run slicing (for Generic surfaces) ----------------
STEP_START_RX = re.compile(r"^(\s*)-\s+\S.*$", re.M)  # any "- <something>" line
RUN_KEY_RX    = re.compile(r"""^(\s*)(run|script)\s*:\s*(\|>|)?\s*$""", re.I | re.M)
LIST_ITEM_RX  = re.compile(r"""^\s*-\s+""")
NAME_KEY_RX   = re.compile(r"""^\s*name\s*:\s*(.+?)\s*$""", re.I)

def _yaml_step_spans(yaml_text: str) -> list[tuple[int,int]]:
    lines = yaml_text.splitlines()
    n = len(lines); spans=[]; i=0
    while i < n:
        m = STEP_START_RX.match(lines[i])
        if not m: i+=1; continue
        step_indent = len(m.group(1) or "")
        j = i+1
        while j < n:
            if STEP_START_RX.match(lines[j]): break
            cur_indent = len(lines[j]) - len(lines[j].lstrip(" "))
            if cur_indent < step_indent and lines[j].strip(): break
            j += 1
        spans.append((i+1, j))
        i = j
    return spans

def _slice_by_lines(text: str, ls: int, le: int) -> str:
    lines = text.splitlines()
    return "\n".join(lines[ls-1:le-1])

WITH_BLOCK_HEAD_RX = re.compile(r"^\s*with\s*:\s*$", re.I|re.M)

def _yaml_with_block_items(yaml_text: str, ls: int, le: int) -> dict:
    sub = _slice_by_lines(yaml_text, ls, le)
    m = WITH_BLOCK_HEAD_RX.search(sub)
    if not m: return {}
    sub_lines = sub.splitlines()
    with_idx = 0
    for idx, ln in enumerate(sub_lines):
        if WITH_BLOCK_HEAD_RX.match(ln): with_idx = idx; break
    out = {}
    with_indent = len(sub_lines[with_idx]) - len(sub_lines[with_idx].lstrip())
    for ln in sub_lines[with_idx+1:]:
        if not ln.strip(): continue
        indent = len(ln) - len(ln.lstrip())
        if indent <= with_indent: break
        if ":" in ln:
            k, v = ln.strip().split(":", 1)
            out[k.strip()] = v.strip().strip("\"'")
    return out

def _yaml_step_name(yaml_text: str, ls: int, le: int) -> Optional[str]:
    sub = _slice_by_lines(yaml_text, ls, le)
    for ln in sub.splitlines():
        m = NAME_KEY_RX.match(ln)
        if m: return m.group(1).strip()
    return None

# ---------------- Orchestrator detection ----------------
ORCH_COORD_RX  = re.compile(r"androidx\.test:orchestrator(?::[^\s'\"\)]+)?", re.I)
ORCH_EXEC_RX   = re.compile(r"testOptions\s*\{[^}]*execution\s*['\"]ANDROIDX_TEST_ORCHESTRATOR['\"]", re.I|re.S)
ORCH_FLAG_RX   = re.compile(r"\buseOrchestrator\s*(?:=|\s)\s*true\b", re.I)
ORCH_PROP_RX   = re.compile(r"\bandroid(?:\.testInstrumentationRunnerArguments)?\.use(?:Test)?Orchestrator\s*=\s*true", re.I)
ORCH_YAML_RX   = re.compile(r"\b(useOrchestrator|orchestrator)\s*[:=]\s*true\b", re.I)

def orchestrator_enabled_in_text(text: str) -> bool:
    t = text or ""
    return any(rx.search(t) for rx in (ORCH_COORD_RX, ORCH_EXEC_RX, ORCH_FLAG_RX, ORCH_PROP_RX, ORCH_YAML_RX))

# ---------------- Test command + runner args ----------------
GRADLE_CMD_LINE_RX = re.compile(r'(^|\n)\s*(?:\./)?gradlew(?:\.bat)?\s+([^\n]+)', re.I)
ADB_INSTR_RX       = re.compile(r'(^|\n)\s*adb\s+(?:-s\s+\S+\s+)?shell\s+am\s+instrument\b([^\n]+)', re.I)

def _find_test_command(text: str) -> Optional[str]:
    m = GRADLE_CMD_LINE_RX.search(text or "")
    if m:
        cmd = m.group(0).strip()
        if re.search(r'\bconnected(Android)?Test\b|\bconnectedCheck\b|\binstrumentation\b|\bandroidTest\b', cmd, re.I):
            return cmd
    m = ADB_INSTR_RX.search(text or "")
    if m:
        return ("adb shell am instrument" + m.group(2)).strip()
    return None

def _parse_runner_args_from_cmd(cmd: str) -> Optional[dict]:
    if not cmd:
        return None
    args: Dict[str, str] = {}
    for key in ("package", "class", "annotation"):
        m = re.search(rf'-Pandroid\.testInstrumentationRunnerArguments\.{key}=([^\s"]+)', cmd)
        if m:
            args[key] = m.group(1)
    if "class" in args and "#" in args["class"]:
        klass, meth = args["class"].split("#", 1)
        args["class"] = klass
        args["method"] = meth
    for key in ("class", "package", "annotation"):
        m = re.search(rf'\s-e\s+{key}\s+([^\s"]+)', cmd)
        if m:
            args[key] = m.group(1)
    if "class" in args and "#" in args["class"]:
        klass, meth = args["class"].split("#", 1)
        args["class"] = klass
        args["method"] = meth
    return args or None

# ---------------- Generic parser (ReactiveCircus + Malinskiy) ----------------
def parse_generic_from_yaml(yaml_text: str) -> list[dict]:
    hits=[]
    for (ls, le) in _yaml_step_spans(yaml_text):
        step_txt = _slice_by_lines(yaml_text, ls, le)
        if not EMULATOR_RUNNER_USES_RX.search(step_txt):
            continue
        kv = _yaml_with_block_items(yaml_text, ls, le) or {}
        step_name = _yaml_step_name(yaml_text, ls, le)

        norm = {
            "emulator_type":"Generic",
            "api_level":None,"image_source":None,"abi":None,"device_profile":None,"device_identifier":None,
            "start_mode":None,"gpu_mode":None,
            "headless_flags":{"no_window":None,"no_audio":None,"no_boot_anim":None},
            "boot_wait": None, "boot_wait_seconds": None,
            "test_command":None,"runner_args":None,
            "context_anchor":f"step:{step_name}" if step_name else "step:<unknown>",
        }

        # ReactiveCircus inputs
        if REACTIVECIRCUS_RX.search(step_txt):
            if "api-level" in kv and RE_INT.search(kv["api-level"]):
                norm["api_level"] = int(RE_INT.search(kv["api-level"]).group())
            if "target" in kv:  norm["image_source"]=kv["target"]
            if "arch" in kv:    norm["abi"]=kv["arch"]
            emu_opts = kv.get("emulator-options","")
            flags_src = emu_opts
        # Malinskiy inputs (compact)
        else:
            if "api" in kv and RE_INT.search(kv["api"]):
                norm["api_level"] = int(RE_INT.search(kv["api"]).group())
            if "tag" in kv:     norm["image_source"]=kv["tag"]
            if "abi" in kv:     norm["abi"]=kv["abi"]
            if "cmd" in kv and kv["cmd"]:
                norm["test_command"] = kv["cmd"]
            flags_src = kv.get("emulator-options", kv.get("emulator-args",""))

        txt = flags_src or step_txt
        norm["headless_flags"]["no_window"]    = bool(re.search(r"-no-window\b", txt))
        norm["headless_flags"]["no_audio"]     = bool(re.search(r"-no-?audio\b", txt))
        norm["headless_flags"]["no_boot_anim"] = bool(re.search(r"-no-boot-anim\b", txt))

        hits.append(norm)
    return hits

# ---------------- GMD (Gradle) parser ----------------
def _find_block_span_from_head(text: str, head_start: int) -> tuple[int,int] | None:
    i = text.find("{", head_start)
    if i == -1: return None
    depth = 0
    for j in range(i, len(text)):
        ch = text[j]
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return (i, j)
    return None

def _char_span_to_line_span(text: str, ci: tuple[int,int]) -> tuple[int,int]:
    start_char, end_char = ci
    start_line = text[:start_char].count("\n") + 1
    end_line   = text[:end_char+1].count("\n") + 1
    return (start_line, end_line)

GMD_BLOCK_HEAD_RX = re.compile(r"(?:\bmanagedDevices\s*\{|testOptions\s*\{[^}]*devices)", re.I|re.S)
GMD_DEVICE_RX = re.compile(r'\bdevice\s*=\s*"([^"]+)"')
GMD_API_RX    = re.compile(r'\bapiLevel\s*=\s*(\d+)')
GMD_SRC_RX    = re.compile(r'\bsystemImageSource\s*=\s*"([^"]+)"')
GMD_BLOCK_NAME_RX = re.compile(r'(\w+)\s*\(\s*com\.android\.build\.api\.dsl\.ManagedVirtualDevice\s*\)', re.I)

def find_gmd_line_spans(gradle_text: str) -> list[tuple[int,int]]:
    spans=[]
    for m in GMD_BLOCK_HEAD_RX.finditer(gradle_text or ""):
        ci = _find_block_span_from_head(gradle_text, m.start())
        if ci: spans.append(_char_span_to_line_span(gradle_text, ci))
    return spans

def parse_gmd_from_gradle(gradle_text: str, span: tuple[int,int]) -> dict:
    lines = (gradle_text or "").splitlines()
    block_text = "\n".join(lines[span[0]-1:span[1]])
    norm = {
        "emulator_type":"GMD",
        "api_level":None,"image_source":None,"abi":None,
        "device_profile":None,"device_identifier":None,
        "start_mode":None,"gpu_mode":None,
        "headless_flags":{"no_window":None,"no_audio":None,"no_boot_anim":None},
        "boot_wait": None, "boot_wait_seconds": None,
        "gmd_device":None,"gmd_api_level":None,"gmd_system_image_source":None,
        "test_command":None,"runner_args":None,"context_anchor":None,
    }
    if (m := GMD_DEVICE_RX.search(block_text)):
        norm["device_profile"]=m.group(1); norm["gmd_device"]=m.group(1)
    if (m := GMD_API_RX.search(block_text)):
        norm["api_level"]=int(m.group(1)); norm["gmd_api_level"]=int(m.group(1))
    if (m := GMD_SRC_RX.search(block_text)):
        norm["image_source"]=m.group(1); norm["gmd_system_image_source"]=m.group(1)
    if (m := GMD_BLOCK_NAME_RX.search(block_text)):
        norm["device_identifier"]=m.group(1)
        norm["context_anchor"]=f"managedDevices.{norm['device_identifier']}"
    return norm

# ---------------- DIY parser + strong-signal threshold ----------------
EMU_START_RX  = re.compile(r'(^|\n)\s*emulator(?:\.exe|\.bat)?\s+-avd\s+(\S+)(?P<rest>[^\n]*)', re.I)
AVD_CREATE_RX = re.compile(
    r'avdmanager\s+create\s+avd\b.*?-n\s+(\S+).*?-(?:k|--package)\s+"?system-images;android-(\d+);([^;]+);([A-Za-z0-9_]+)"?(?:.*?-(?:d|--device)\s+(\S+))?',
    re.I
)
SDKIMG_RX     = re.compile(r'system-images;android-(\d+);([^;]+);([A-Za-z0-9_]+)')
BOOT_WAIT_RX  = re.compile(r'adb\s+(?:-s\s+\S+\s+)?wait-for-device|adb\s+shell\s+getprop\s+sys\.boot_completed', re.I)
SLEEP_SEC_RX  = re.compile(r'\bsleep\s+(\d+)\b', re.I)
TIMEOUT_SEC_RX= re.compile(r'\btimeout\s+(?:/t\s+)?(\d+)\b', re.I)
TIMEOUT_MIN_RX= re.compile(r'\btimeout-minutes\s*:\s*(\d+)\b', re.I)

DIY_STRONG_PATTERNS = [
    re.compile(r'(?mi)^[^\n]*\bemulator\b[^\n]*(?:-avd\s+\S+|@\S+)'),
    re.compile(r'(?mi)\bavdmanager\s+create\s+avd\b'),
    re.compile(r'(?mi)\bsdkmanager\b[^\n]*system-images;android-\d+'),
    re.compile(r'(?mi)^\s*(?:\./)?android-wait-for-emulator\b'),
    re.compile(r'(?mi)^\s*adb\s+(?:-s\s+\S+\s+)?wait[- ]?for[- ]?device\b'),
]
DIY_CORROBORATORS = [
    re.compile(r'-no-window|-no-?audio|-no-boot-anim', re.I),
    re.compile(r'\bsleep\s+\d+\b|\btimeout-minutes\s*:\s*\d+\b', re.I),
    re.compile(r'getprop\s+sys\.boot_completed', re.I),
]

def _extract_wait_seconds(text: str) -> Optional[int]:
    secs=[]
    for rx in (SLEEP_SEC_RX, TIMEOUT_SEC_RX):
        for m in rx.finditer(text or ""):
            try: secs.append(int(m.group(1)))
            except: pass
    for m in TIMEOUT_MIN_RX.finditer(text or ""):
        try: secs.append(int(m.group(1))*60)
        except: pass
    if secs: return max(secs)
    return None

def parse_diy_from_text(text: str) -> dict:
    t = text or ""
    norm = {
        "emulator_type":"DIY",
        "api_level":None,"image_source":None,"abi":None,
        "device_profile":None,"device_identifier":None,
        "start_mode":None,"gpu_mode":None,
        "headless_flags":{"no_window":None,"no_audio":None,"no_boot_anim":None},
        "boot_wait": None, "boot_wait_seconds": None,
        "test_command":None,"runner_args":None,"context_anchor":None,
    }
    anchor=None
    if (m := AVD_CREATE_RX.search(t)):
        norm["device_identifier"]=m.group(1)
        norm["api_level"]=int(m.group(2))
        norm["image_source"]=m.group(3)
        norm["abi"]=m.group(4)
        if m.group(5): norm["device_profile"]=m.group(5)
        anchor=f"avdmanager create:{norm['device_identifier']}"
    elif (m := SDKIMG_RX.search(t)):
        norm["api_level"]=int(m.group(1)); norm["image_source"]=m.group(2); norm["abi"]=m.group(3)
        anchor="sdkmanager system-image"

    if (m := EMU_START_RX.search(t)):
        rest = m.group("rest") or ""
        gm = re.search(r"-(?:-)?gpu\s+(\S+)", rest); norm["gpu_mode"]=gm.group(1) if gm else None
        norm["headless_flags"]["no_window"]    = bool(re.search(r"-no-window\b", rest))
        norm["headless_flags"]["no_audio"]     = bool(re.search(r"-no-?audio\b", rest))
        norm["headless_flags"]["no_boot_anim"] = bool(re.search(r"-no-boot-anim\b", rest))
        norm["start_mode"] = "cold" if re.search(r"-no-snapshot\b", rest) else "snapshot"
        if not anchor:
            norm["device_identifier"] = norm["device_identifier"] or m.group(2)
            anchor = f"emulator -avd:{m.group(2)}"

    boot_wait = bool(BOOT_WAIT_RX.search(t))
    norm["boot_wait"] = boot_wait
    if boot_wait:
        norm["boot_wait_seconds"] = _extract_wait_seconds(t)
    norm["context_anchor"] = anchor or "diy:<unknown>"
    return norm

# ---------------- Host OS infer ----------------
def infer_host_from_text(text: str) -> tuple[Optional[str], Optional[str]]:
    t = (text or "").lower()
    if "ubuntu" in t or "runs-on: ubuntu" in t or "linux" in t:  return "linux","kvm"
    if "macos" in t or "runs-on: macos" in t or "macos-latest" in t or "mac " in t: return "macos","hvf"
    return None, None

# ---------------- Labels/helpers ----------------
def file_kind_for_path(p: str) -> str:
    if is_gradle_file(p): return "gradle"
    if is_ci_file(p) and p.lower().endswith((".yml",".yaml")): return "ci_yaml"
    if is_script_file(p): return "script"
    return "other"

DIFF_HUNK_RX = re.compile(r"@@ -\d+(?:,\d+)? \+(\d+)(?:,(\d+))? @@")

def added_line_numbers_for_path(repo_dir: Path, sha: str, path: str) -> Optional[set[int]]:
    try:
        cp = sh(["git","-c","core.quotepath=off","diff","-U0", f"{sha}^", sha, "--", path],
                cwd=repo_dir, check=False)
    except Exception:
        return None
    if cp.returncode not in (0,1): return None
    added:set[int]=set(); new_line=None
    for line in cp.stdout.splitlines():
        m = DIFF_HUNK_RX.match(line)
        if m:
            start=int(m.group(1)); length=int(m.group(2) or "1"); new_line=start; continue
        if new_line is None: continue
        if line.startswith("+") and not line.startswith("+++"):
            added.add(new_line); new_line += 1
        elif line.startswith("-") and not line.startswith("---"):
            pass
        else:
            if not (line.startswith("---") or line.startswith("+++")):
                new_line += 1
    return added

def find_emulator_run_spans(yaml_text: str) -> list[tuple[int,int]]:
    lines = yaml_text.splitlines(); n=len(lines); spans=[]; i=0
    while i < n:
        mstep = STEP_START_RX.match(lines[i])
        if not mstep: i+=1; continue
        step_indent = len(mstep.group(1) or ""); j = i+1
        while j < n:
            if STEP_START_RX.match(lines[j]): break
            cur_indent = len(lines[j]) - len(lines[j].lstrip(" "))
            if cur_indent < step_indent and lines[j].strip(): break
            mrun = RUN_KEY_RX.match(lines[j])
            if mrun:
                block_indent = len(mrun.group(1) or ""); k = j+1; block_idx=[]
                while k < n:
                    ln = lines[k]
                    if not ln.strip(): block_idx.append(k); k+=1; continue
                    ind = len(ln) - len(ln.lstrip(" "))
                    if ind <= block_indent and not LIST_ITEM_RX.match(ln): break
                    block_idx.append(k); k+=1
                block_text = "\n".join(lines[x] for x in block_idx)
                if DIY_RX.search(block_text) or EMULATOR_RUNNER_USES_RX.search(block_text):
                    if block_idx: spans.append((min(block_idx)+1, max(block_idx)+1))
                j = k; continue
            j += 1
        i = j
    # merge adjacent
    merged: list[list[int]]=[]
    for s,e in sorted(spans):
        if not merged or s > merged[-1][1] + 1: merged.append([s,e])
        else: merged[-1][1] = max(merged[-1][1], e)
    return [(s,e) for s,e in merged]

def lines_overlap_spans(added: set[int] | None, spans: list[tuple[int,int]]) -> Optional[bool]:
    if added is None: return None
    if not spans: return False
    for a in added:
        for s,e in spans:
            if s <= a <= e: return True
    return False

# ---------------- Medium/toolchain parsers ----------------
def parse_medium_from_context(text: str) -> dict:
    t = text or ""
    out = {
        "image_flavor": None, "orchestrator_enabled": None, "num_shards": None,
        "locale": None, "timezone": None, "network_profile": None,
        "gmd_device": None, "gmd_api_level": None, "gmd_system_image_source": None,
    }
    if (m := re.search(r'\b(numShards|shards?)\s*[:=]\s*(\d+)', t, re.I)): out["num_shards"] = int(m.group(2))
    if (m := re.search(r'\blocale\s*[:=]\s*([A-Za-z]{2,3}[-_][A-Za-z]{2,3})', t)): out["locale"] = m.group(1)
    if (m := re.search(r'\b(timezone|TZ)\s*[:=]\s*([A-Za-z_/\-]+)', t)): out["timezone"] = m.group(2)
    if (m := re.search(r'\b(networkProfile|throttle(?:d|))\s*[:=]\*?([A-Za-z0-9_\-]+)', t, re.I)): out["network_profile"] = m.group(2)
    if re.search(r'\bplay\b', t, re.I): out["image_flavor"]="play"
    elif re.search(r'google-?apis', t, re.I): out["image_flavor"]="google_apis"
    elif re.search(r'google-?atd', t, re.I): out["image_flavor"]="google-atd"
    elif re.search(r'\baosp\b', t, re.I): out["image_flavor"]="aosp"
    if orchestrator_enabled_in_text(t): out["orchestrator_enabled"]=True
    return out

def parse_toolchain_from_context(path: str, text: str) -> dict:
    t = text or ""; out = {
        "agp_version": None, "gradle_version": None, "jdk_version": None,
        "android_sdk_installed": None, "android_sdk_licenses": None,
    }
    if (m := re.search(r'com\.android\.tools\.build:gradle:([0-9][^\'"\s\)]+)', t)): out["agp_version"]=m.group(1)
    if "gradle/wrapper/gradle-wrapper.properties" in path.replace("\\","/"):
        if (m := re.search(r"distributionUrl=.*?/gradle-([0-9][\w\.\-]+)-", t)): out["gradle_version"]=m.group(1)
    if (m := re.search(r'(java-version|actions/setup-java).*(?:=|:)\s*([0-9][0-9]?)', t, re.I)): out["jdk_version"]=m.group(2)
    if re.search(r'\bsdkmanager\b', t): out["android_sdk_installed"]=True
    if re.search(r'sdkmanager\s+--licenses', t): out["android_sdk_licenses"]=True
    return out

def write_jsonl(path: Path, rows: List[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

# ---------------- NEW: quick HEAD scan for audit (more tolerant) ----------------
def _slurp(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="replace")
    except Exception:
        return ""

def _looks_ci_yaml(p: Path) -> bool:
    s = str(p).replace("\\","/")
    return is_ci_file(s) and s.lower().endswith((".yml",".yaml"))

def _looks_gradle(p: Path) -> bool:
    return is_gradle_file(str(p))

def _looks_script(p: Path) -> bool:
    return is_script_file(str(p))

def head_has_any_emulator(repo_dir: Path) -> bool:
    """
    Returns True if we can detect any emulator style (Generic/DIY/GMD) in HEAD.
    More tolerant: also search for runner action strings even without 'uses:' prefix.
    """
    try:
        candidates: list[Path] = []
        for root, dirs, files in os.walk(repo_dir):
            bn = os.path.basename(root).lower()
            if bn in {".git", "build", ".gradle", "node_modules"}:
                continue
            for fn in files:
                p = Path(root) / fn
                if _looks_ci_yaml(p) or _looks_gradle(p) or _looks_script(p):
                    candidates.append(p)

        for p in candidates:
            txt = _slurp(p)
            if p.suffix.lower() in (".yml", ".yaml"):
                # tolerant quick checks
                if re.search(r"android-emulator-runner@[\w.\-]+", txt or "", re.I) \
                   or re.search(r"action-android/emulator-run-cmd@[\w.\-]+", txt or "", re.I):
                    return True
                # structured parse (quoted uses supported)
                if EMULATOR_RUNNER_USES_RX.search(txt or "") and parse_generic_from_yaml(txt):
                    return True
            if _looks_gradle(p):
                if find_gmd_line_spans(txt or ""):
                    return True
            if _looks_script(p) or _looks_ci_yaml(p):
                if DIY_RX.search(txt or "") and any(rx.search(txt or "") for rx in DIY_STRONG_PATTERNS):
                    return True
        return False
    except Exception:
        return False

# ---------------- main ----------------
def main():
    repos = [p for p in CLONE_ROOT.iterdir() if (p / ".git").exists()]
    repos.sort(key=lambda p: p.name.lower())
    print(f"Found {len(repos)} repos in {CLONE_ROOT}")

    ok=skip=err=0
    audit_rows: list[dict] = []

    for repo in repos:
        try:
            rel_commits = list_relevant_commits(repo)
            if MAX_COMMITS_PER_REPO > 0:
                rel_commits = rel_commits[:MAX_COMMITS_PER_REPO]

            out_rows: List[dict] = []
            # per-repo audit accumulators
            any_snapshot = False
            any_param_change = False
            any_detected_params = False
            snapshot_commit_shas: set[str] = set()

            for sha, ts, changed_paths in rel_commits:
                rel_paths = [p for p in changed_paths if is_ci_file(p) or is_gradle_file(p) or is_script_file(p)]
                if not rel_paths:
                    continue

                subj = git_subject(repo, sha)
                commit_had_snapshot = False

                for pth in rel_paths:
                    txt = git_show(repo, sha, pth)
                    if txt is None:
                        continue

                    detected: list[dict] = []
                    test_cmd = _find_test_command(txt)
                    runner_args = _parse_runner_args_from_cmd(test_cmd or "")

                    # --- Generic (ReactiveCircus + Malinskiy)
                    if pth.lower().endswith((".yml",".yaml")):
                        for g in parse_generic_from_yaml(txt):
                            g["test_command"]=g.get("test_command") or test_cmd
                            g["runner_args"]=runner_args
                            detected.append(g)

                    # --- GMD in Gradle
                    if is_gradle_file(pth):
                        for sp in find_gmd_line_spans(txt):
                            gmd = parse_gmd_from_gradle(txt, sp)
                            gmd["test_command"]=test_cmd
                            gmd["runner_args"]=runner_args
                            detected.append(gmd)

                    # --- DIY in scripts OR ANY CI YAML (strong-signal threshold)
                    is_yaml = pth.lower().endswith((".yml",".yaml"))
                    if is_script_file(pth) or (is_yaml and is_ci_file(pth)):
                        if DIY_RX.search(txt or ""):
                            strong = any(rx.search(txt or "") for rx in DIY_STRONG_PATTERNS)
                            if strong:
                                diy = parse_diy_from_text(txt or "")
                                diy["test_command"]=test_cmd
                                diy["runner_args"]=runner_args
                                if any(diy.get(k) for k in (
                                        "api_level","image_source","abi",
                                        "device_identifier","gpu_mode","boot_wait","boot_wait_seconds"
                                    )) or strong:
                                    detected.append(diy)

                    feats: Dict[str,Any] = {}
                    if detected:
                        any_detected_params = True
                        host_os, hypervisor = infer_host_from_text(txt)
                        for dp in detected:
                            dp.setdefault("host_os", host_os)
                            dp.setdefault("hypervisor", hypervisor)
                        feats["detected_params"] = detected

                    # Medium & toolchain
                    med = parse_medium_from_context(txt or "")
                    tool = parse_toolchain_from_context(pth, txt or "")
                    if any(v is not None for v in med.values()):
                        feats["medium_params"] = {k:v for k,v in med.items() if v is not None}
                    if any(v is not None for v in tool.values()):
                        feats["toolchain_params"] = {k:v for k,v in tool.items() if v is not None}

                    # Labels
                    fk = file_kind_for_path(pth)
                    added_lines = added_line_numbers_for_path(repo, sha, pth)
                    ci_spans  = find_emulator_run_spans(txt) if fk == "ci_yaml" else None
                    gmd_spans = find_gmd_line_spans(txt)     if fk == "gradle"  else None
                    ci_emu_change     = lines_overlap_spans(added_lines, ci_spans or []) if fk == "ci_yaml" else None
                    gradle_gmd_change = lines_overlap_spans(added_lines, gmd_spans or []) if fk == "gradle" else None

                    labels = {
                        "file_kind": fk,
                        "ci_emulator_step_change": ci_emu_change,
                        "gradle_gmd_block_change": gradle_gmd_change,
                        "gmd_present": bool(gmd_spans) if fk == "gradle" else None,
                    }
                    feats["labels"] = labels

                    if not feats:
                        continue

                    commit_had_snapshot = True
                    any_snapshot = True
                    if (ci_emu_change is True) or (gradle_gmd_change is True):
                        any_param_change = True

                    # ---------- APPEND CONTENT (minimal addition) ----------
                    row = {
                        "repo": repo.name,
                        "sha": sha,
                        "timestamp": ts,
                        "subject": subj,
                        "path": pth,
                        "features": feats,
                        "content": txt,  # <--- ADDED: include file text so CCE can be generated
                    }
                    if INCLUDE_LANGUAGE:
                        row["language"] = infer_lang(pth)
                    if INCLUDE_SIZE:
                        row["size"] = len(txt)

                    out_rows.append(row)

                if commit_had_snapshot:
                    snapshot_commit_shas.add(sha)

            # Write JSONL if we collected any
            if not out_rows:
                print(f"[skip] {repo.name}: no relevant snapshots")
            else:
                dst = SNAPSHOT_DIR / f"{repo.name}.jsonl"
                write_jsonl(dst, out_rows)
                print(f"[ok] {repo.name}: {len(out_rows)} snapshots -> {dst}"); ok += 1

            # --- Per-repo audit summary row (with commit count) ---
            emulator_on_head = head_has_any_emulator(repo)
            emulator_detected = (emulator_on_head or any_detected_params)

            audit_rows.append({
                "repo": repo.name,
                "emulator_detected": "Yes" if emulator_detected else "No",
                "detected_any_commit": "Yes" if any_snapshot else "No",
                "detected_any_param_change": "Yes" if any_param_change else "No",
                "detected_commit_count": len(snapshot_commit_shas),
            })

            if not out_rows:
                skip += 1

        except Exception as e:
            print(f"[err] {repo.name}: {e}"); err += 1

    # --- Write audit CSV ---
    if audit_rows:
        AUDIT_CSV.parent.mkdir(parents=True, exist_ok=True)
        import csv as _csv
        with AUDIT_CSV.open("w", newline="", encoding="utf-8") as f:
            w = _csv.DictWriter(
                f,
                fieldnames=["repo","emulator_detected","detected_any_commit",
                            "detected_any_param_change","detected_commit_count"]
            )
            w.writeheader(); w.writerows(audit_rows)
        print(f"\nAudit CSV written: {AUDIT_CSV}")

    print(f"\nDone. ok={ok}, skip={skip}, err={err}, out_dir={SNAPSHOT_DIR}")

if __name__ == "__main__":
    main()


[cutoff] Using commit cutoff <= 2025-08-10 23:59:59 -0400 (epoch=1754884799)
Found 9 repos in C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV8.1
[ok] Arian04__android-hid-client: 98 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshotsV8.1\Arian04__android-hid-client.jsonl
[ok] arkivanov__android-dev-challenge-compose-2: 8 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshotsV8.1\arkivanov__android-dev-challenge-compose-2.jsonl
[ok] elftausend__custos: 71 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshotsV8.1\elftausend__custos.jsonl
[ok] guolindev__android-dev-challenge-compose: 11 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshotsV8.1\guolindev__android-dev-challenge-compose.jsonl
[ok] ICBNetwork__web3-Wallet: 27 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshotsV8.1\ICBNetwork__web3-Wallet.jsonl
[ok] okanaydin__Android-Compose-Ch

## Step 3: labeling

In [ ]:
# RQ2 — Step 3: Labelling, V8.1 (aligned with Step-1 v8.1)
# - NO baseline rows (emit only when a real delta exists)
# - Proper change_type: added / removed / modified
# - Proper change_op: add / remove / value_edit / no_change
# - Reads Step-1 detected_params[*].emulator_type -> episode style Old/New/Change
# - Tracks GMD & Orchestrator presence direction
# - Computes runtime wait deltas using boot_wait_seconds (new in v8.x miner)
# - Updated CI/vendor patterns and generic CI dirs to match Step-1
# - Updated paths: snapshotsV8.1 -> cce_enrichedV8.1

from __future__ import annotations
import json, re
import datetime as _dt
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

# -----------------------------
# Config
# -----------------------------
WORK_ROOT         = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
SNAPSHOT_DIR      = WORK_ROOT / "snapshotsV8.1"          # updated
OUT_DIR           = WORK_ROOT / "cce_enrichedV8.1"       # updated
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Small helpers
# -----------------------------
VERSION_RE = re.compile(r"\d+(?:\.\d+)*")

def read_snapshots(folder: Path) -> Dict[str, List[dict]]:
    by_repo: Dict[str, List[dict]] = {}
    for p in folder.glob("*.jsonl"):
        with p.open(encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                d = json.loads(line)
                repo = d.get("repo")
                if not repo:
                    continue
                by_repo.setdefault(repo, []).append(d)
    # preserve per-path chronological order
    for repo, rows in by_repo.items():
        rows.sort(key=lambda r: (r.get("path",""), int(r.get("timestamp", 0)), r.get("sha","")))
    return by_repo

def stringify(x: Any) -> str:
    if isinstance(x, (dict, list, set, tuple)):
        try: return json.dumps(x, ensure_ascii=False, sort_keys=True)
        except: return str(x)
    return "" if x is None else str(x)

def _safe_json_loads(s: str):
    try: return json.loads(s) if s else None
    except: return None

def ensure_utc_epoch(ts_any) -> int:
    try: ts = int(float(ts_any))
    except: ts = 0
    return ts

def epoch_to_iso_utc(ts: int) -> str:
    return _dt.datetime.fromtimestamp(int(ts), tz=_dt.timezone.utc).isoformat().replace("+00:00", "Z")

def parse_version_tuple(s: str) -> Tuple[int, ...]:
    m = VERSION_RE.search(str(s) if s is not None else "")
    if not m: return tuple()
    parts = m.group(0).split(".")
    out: List[int] = []
    for p in parts:
        try: out.append(int(p))
        except: out.append(0)
    return tuple(out)

def max_version_tuple(strings: List[str]) -> Tuple[int, ...]:
    best: Tuple[int, ...] = tuple()
    for s in strings or []:
        vt = parse_version_tuple(str(s))
        if vt > best: best = vt
    return best

def _to_bool_or_none(x) -> Optional[bool]:
    if isinstance(x, bool): return x
    if isinstance(x, (int, float)): return bool(int(x))
    if isinstance(x, str):
        s = x.strip().lower()
        if s in {"true","1","yes"}: return True
        if s in {"false","0","no"}: return False
    return None

def _max_int_or_none(vals: List[str]) -> Optional[int]:
    best = None
    for v in vals or []:
        try:
            i = int(str(v).strip())
            best = i if best is None else max(best, i)
        except:
            continue
    return best

# -----------------------------
# Study-defined fields (What to diff)
# -----------------------------
COVERAGE_FIELDS = ["api_level","image_source","abi","device_profile","device_identifier"]
RUNTIME_FIELDS  = ["start_mode","gpu_mode","headless_flags","boot_wait","boot_wait_seconds"]  # added boot_wait_seconds
HOST_FIELDS     = ["host_os","hypervisor"]
MEDIUM_FIELDS   = ["image_flavor","orchestrator_enabled","num_shards","locale","timezone","network_profile"]
TOOLCHAIN_FIELDS = ["agp_version","jdk_version"]
ALL_FIELDS_ORDER = COVERAGE_FIELDS + RUNTIME_FIELDS + HOST_FIELDS + MEDIUM_FIELDS + TOOLCHAIN_FIELDS

# -----------------------------
# Primary label mapping (WHAT) → Category
# -----------------------------
CAT_COVERAGE = "Coverage"
CAT_RUNTIME  = "Runtime"
CAT_CONTEXT  = "Context"
CAT_INVOC    = "Invocation"   # retained for completeness

FIELD_TO_PRIMARY = {
    # Coverage
    "api_level":          "api_bump",
    "image_source":       "system_image_change",
    "abi":                "abi_change",
    "device_profile":     "device_profile_change",
    "device_identifier":  "device_profile_change",

    # Runtime
    "start_mode":         "wait_strategy_change",
    "gpu_mode":           "timeout_tuning",
    "headless_flags":     "timeout_tuning",
    "boot_wait":          "wait_strategy_change",
    "boot_wait_seconds":  "wait_strategy_change",   # new mapping

    # Host / Context
    "host_os":            "runner_os_change",
    "hypervisor":         "runner_os_change",

    # Medium (context)
    "image_flavor":           "runner_os_change",
    "orchestrator_enabled":   "runner_os_change",
    "num_shards":             "runner_os_change",
    "locale":                 "runner_os_change",
    "timezone":               "runner_os_change",
    "network_profile":        "runner_os_change",

    # Toolchain
    "agp_version":        "agp_bump",
    "jdk_version":        "jdk_bump",
}

PRIMARY_TO_CATEGORY = {
    "api_bump":              CAT_COVERAGE,
    "system_image_change":   CAT_COVERAGE,
    "abi_change":            CAT_COVERAGE,
    "device_profile_change": CAT_COVERAGE,

    "wait_strategy_change":  CAT_RUNTIME,
    "timeout_tuning":        CAT_RUNTIME,

    "runner_os_change":      CAT_CONTEXT,
    "agp_bump":              CAT_CONTEXT,
    "jdk_bump":              CAT_CONTEXT,
    "other_change":          CAT_CONTEXT,
}

CATEGORY_NAME = {
    CAT_COVERAGE: "Coverage",
    CAT_RUNTIME:  "Runtime",
    CAT_CONTEXT:  "Context",
    CAT_INVOC:    "Invocation",
}

# -----------------------------
# Emulator style helpers (episode-level)
# -----------------------------
def _norm_style(s: Optional[str]) -> Optional[str]:
    if not s: return None
    t = str(s).strip().lower()
    if t in {"generic", "diy", "gmd"}:
        return t
    return None

def _style_from_dps(dps: list[dict]) -> Optional[str]:
    """Collapse per-dp emulator_type -> one snapshot style. If multiple different styles, return 'mixed'."""
    seen = { _norm_style(dp.get("emulator_type")) for dp in (dps or []) }
    seen.discard(None)
    if not seen: return None
    if len(seen) == 1: return next(iter(seen))
    return "mixed"

# -----------------------------
# Aggregate snapshot → set-of-values per field
# -----------------------------
def aggregate_snapshot(feats: dict) -> Dict[str, set]:
    agg: Dict[str, set] = {k: set() for k in ALL_FIELDS_ORDER}
    dps = (feats or {}).get("detected_params") or []

    for dp in dps:
        for f in COVERAGE_FIELDS + RUNTIME_FIELDS + HOST_FIELDS:
            v = dp.get(f)
            if v is None:
                continue
            if f == "headless_flags" and isinstance(v, dict):
                token = f"nw{int(bool(v.get('no_window')))}_na{int(bool(v.get('no_audio')))}_nba{int(bool(v.get('no_boot_anim')))}"
                agg["headless_flags"].add(token)
            else:
                # keep ints (e.g., boot_wait_seconds) comparable by storing as strings
                agg[f].add(str(v) if not isinstance(v, (bool, int)) else str(v))

    # medium/toolchain
    med = (feats or {}).get("medium_params") or {}
    for f in MEDIUM_FIELDS:
        if med.get(f) is not None:
            vv = med.get(f)
            agg[f].add(str(vv) if not isinstance(vv, (bool,int)) else str(vv))

    tool = (feats or {}).get("toolchain_params") or {}
    for f in TOOLCHAIN_FIELDS:
        if tool.get(f) is not None:
            agg[f].add(str(tool.get(f)))

    return agg

def _labels(feats: dict) -> dict:
    return (feats or {}).get("labels", {}) or {}

def _gmd_present(feats: dict) -> Optional[bool]:
    """Prefer explicit Step-1 bit if available; else None."""
    lbl = _labels(feats)
    if "gmd_present" in lbl:
        return _to_bool_or_none(lbl.get("gmd_present"))
    return None

def _orch_present_boolset(agg_map: Dict[str,set]) -> Optional[bool]:
    """Collapse orchestrator_enabled set into a single boolean if unambiguous."""
    vals = list(agg_map.get("orchestrator_enabled", set()))
    norm = [_to_bool_or_none(v) for v in vals]
    norm = [v for v in norm if v is not None]
    if not norm:
        return None
    return norm[0] if len(set(norm)) == 1 else None

# -----------------------------
# Diff engine → per-field CCEs (with proper change_type)
# -----------------------------
def diff_features(old_feats: Dict[str, Any], new_feats: Dict[str, Any]) -> Tuple[List[Dict[str, Any]], dict]:
    """
    Return (list_of_field_deltas, episode_aux)
    episode_aux holds episode-level evidence: GMD/Orchestrator presence and Emulator Style Old/New
    """
    oldm = aggregate_snapshot(old_feats or {})
    newm = aggregate_snapshot(new_feats or {})

    out: List[Dict[str, Any]] = []

    def handle_field(field: str):
        a = oldm.get(field, set())
        b = newm.get(field, set())
        if a == b:
            return

        added   = sorted(b - a)
        removed = sorted(a - b)

        # Proper change_type
        if not a and b:
            change_type = "added"
        elif a and not b:
            change_type = "removed"
        else:
            change_type = "modified"

        row = {
            "field": field,
            "old_value": stringify(sorted(a)),
            "new_value": stringify(sorted(b)),
            "change_type": change_type,
        }
        if added:   row["added_items"] = stringify(added)
        if removed: row["removed_items"] = stringify(removed)

        # Magnitude for api_level
        if field == "api_level" and a and b:
            try:
                row["magnitude"] = max(int(x) for x in b) - max(int(x) for x in a)
            except:
                pass

        # Coverage net counts
        if field in COVERAGE_FIELDS:
            try:
                old_list = _safe_json_loads(row["old_value"]) or []
                new_list = _safe_json_loads(row["new_value"]) or []
                added_list = _safe_json_loads(row.get("added_items","")) or []
                removed_list = _safe_json_loads(row.get("removed_items","")) or []
                row["Coverage_Added_Count"] = len(added_list)
                row["Coverage_Removed_Count"] = len(removed_list)
                row["Coverage_Net"] = len(new_list) - len(old_list)
            except:
                row["Coverage_Added_Count"] = row["Coverage_Removed_Count"] = row["Coverage_Net"] = None

        # Runtime wait tuning — use boot_wait_seconds (numeric)
        if field == "boot_wait_seconds":
            try:
                old_list = _safe_json_loads(row["old_value"]) or []
                new_list = _safe_json_loads(row["new_value"]) or []
                old_max = _max_int_or_none(old_list)
                new_max = _max_int_or_none(new_list)
                if old_max is not None and new_max is not None:
                    delta = new_max - old_max
                    row["Runtime_Wait_Delta"] = delta
                    row["Runtime_Wait_Change"] = "increase" if delta > 0 else ("decrease" if delta < 0 else "")
                else:
                    row["Runtime_Wait_Delta"] = None
                    row["Runtime_Wait_Change"] = ""
            except:
                row["Runtime_Wait_Delta"] = None
                row["Runtime_Wait_Change"] = ""

        out.append(row)

    for f in ALL_FIELDS_ORDER:
        handle_field(f)

    # Episode-level AUX: GMD presence & Orchestrator enabled
    gmd_old = _gmd_present(old_feats)
    gmd_new = _gmd_present(new_feats)
    orch_old = _orch_present_boolset(oldm)
    orch_new = _orch_present_boolset(newm)

    # Episode-level emulator style from detections
    style_old = _style_from_dps((old_feats or {}).get("detected_params") or [])
    style_new = _style_from_dps((new_feats or {}).get("detected_params") or [])

    episode_aux = {
        "GMD_Present_Old": gmd_old,
        "GMD_Present_New": gmd_new,
        "Orchestrator_Enabled_Old": orch_old,
        "Orchestrator_Enabled_New": orch_new,
        "Emulator_Style_Old": style_old,   # 'generic' | 'diy' | 'gmd' | 'mixed' | None
        "Emulator_Style_New": style_new,
    }
    return out, episode_aux

# -----------------------------
# Delta → intents (meaningful knobs only)
# -----------------------------
DELTA_MEANINGFUL_FIELDS = set(COVERAGE_FIELDS) | {"agp_version","jdk_version"}  # runtime waits excluded by design

def derive_delta_intents(deltas: List[Dict[str, Any]]) -> List[str]:
    labs: set[str] = set()
    for d in deltas:
        field = d.get("field")
        if field not in DELTA_MEANINGFUL_FIELDS:
            continue
        added = _safe_json_loads(d.get("added_items") or "") or []
        removed = _safe_json_loads(d.get("removed_items") or "") or []

        if field in COVERAGE_FIELDS:
            old_list = _safe_json_loads(d.get("old_value") or "") or []
            new_list = _safe_json_loads(d.get("new_value") or "") or []
            if old_list and new_list:
                if added:   labs.add("expand_coverage")
                if removed: labs.add("reduce_coverage")

        if field in {"agp_version","jdk_version"}:
            old_list = _safe_json_loads(d.get("old_value") or "") or []
            new_list = _safe_json_loads(d.get("new_value") or "") or []
            old_max = max_version_tuple(old_list)
            new_max = max_version_tuple(new_list)
            if new_max > old_max:
                labs.add("version_change")

    return sorted(labs)

# -----------------------------
# Path detectors (generic + field-aware) — aligned w/ Step-1
# -----------------------------
CI_VENDOR_PATTERNS = [
    (re.compile(r'(?i)(?:^|/)\.travis\.ya?ml$'),               'Travis_CI'),
    (re.compile(r'(?i)(?:^|/)\.appveyor\.ya?ml$'),             'AppVeyor'),
    (re.compile(r'(?i)(?:^|/)appveyor\.ya?ml$'),               'AppVeyor'),
    (re.compile(r'(?i)(?:^|/)circle\.yml$'),                   'Circle_CI'),
    (re.compile(r'(?i)(?:^|/)\.circleci/config\.ya?ml$'),      'Circle_CI'),
    (re.compile(r'(?i)(?:^|/)azure-pipelines\.ya?ml$'),        'Azure_Pipelines'),
    (re.compile(r'(?i)(?:^|/)\.github/workflows/.*\.(yml|yaml)$'), 'GitHub_Actions'),
    (re.compile(r'(?i)(?:^|/)bitbucket-pipelines\.ya?ml$'),    'Bitbucket'),
    (re.compile(r'(?i)(?:^|/)\.gitlab-ci\.ya?ml$'),            'GitLab'),
    (re.compile(r'(?i)(?:^|/)Jenkinsfile(?:\.\w+)?$'),         'Jenkins'),
    (re.compile(r'(?i)(?:^|/)bitrise\.ya?ml$'),                'Bitrise'),
    (re.compile(r'(?i)(?:^|/)bamboo\.ya?ml$'),                 'Bamboo'),
    (re.compile(r'(?i)(?:^|/)codeship-services\.ya?ml$'),      'Codeship'),
    (re.compile(r'(?i)(?:^|/)\.gocd\.ya?ml$'),                 'GoCD'),
    (re.compile(r'(?i)(?:^|/)\.cirrus\.ya?ml$'),               'Cirrus_CI'),
    (re.compile(r'(?i)(?:^|/)wercker\.ya?ml$'),                'Wercker'),
    (re.compile(r'(?i)(?:^|/)\.semaphore\.ya?ml$'),            'Semaphore'),
    (re.compile(r'(?i)(?:^|/)codemagic\.ya?ml$'),              'Nevercode'),
]
GENERIC_CI_DIRS   = re.compile(r'(?i)(?:^|/)(?:ci|\.ci|\.jenkins)(?:/|$)')  # added .jenkins

_GRADLE_PATH_RX   = re.compile(
    r"(?i)(?:^|/)(?:build|settings)\.gradle(?:\.kts)?$|(?:^|/)gradle\.properties$|(?:^|/)gradle/wrapper/gradle-wrapper\.properties$"
)
_SCRIPT_PATH_RX   = re.compile(
    r"(?i)\.(?:sh|bash|zsh|py|bat|cmd|ps1|psm1|psd1|ksh)$|(?:^|/)(?:scripts?|tools?)(?:/|$)"
)

def detect_intent_path_generic(path: str) -> List[str]:
    intents: List[str] = []
    p = path or ""
    if any(rx.search(p) for rx, _ in CI_VENDOR_PATTERNS) or GENERIC_CI_DIRS.search(p):
        intents.append("ci_env_workflow")
    if _GRADLE_PATH_RX.search(p):
        intents.append("version_change")
    if _SCRIPT_PATH_RX.search(p):
        intents.append("ci_env_workflow")
    return sorted(set(intents))

def detect_intent_path_field_aware(path: str, field: str) -> Tuple[List[str], Optional[str]]:
    p = path or ""; f = (field or "").lower()
    intents: List[str] = []; reasons: List[str] = []
    vendor = None
    for rx, name in CI_VENDOR_PATTERNS:
        if rx.search(p):
            vendor = name; break
    if vendor is None and GENERIC_CI_DIRS.search(p):
        vendor = "Generic_CI"

    is_gradle = bool(_GRADLE_PATH_RX.search(p))

    if vendor and f in set(COVERAGE_FIELDS):
        intents += ["coverage", "ci_env_workflow"]; reasons.append(f"ci vendor={vendor} + coverage field")
    if is_gradle and f in {"agp_version","jdk_version"}:
        intents.append("version_change"); reasons.append("gradle/build tool path")

    intents = sorted(set(intents))
    return intents, ("; ".join(reasons) if reasons else None)

# -----------------------------
# Subject intents (simple keywords)
# -----------------------------
def detect_subject_intents(subject: str) -> Tuple[List[str], Optional[str]]:
    s = (subject or "").lower()
    hits = set()
    if any(w in s for w in ["flake","flaky","retry","retries","timeout","timeouts","stabil","crash","fix"]):
        hits.add("flake_mitigation")
    if any(w in s for w in ["speed up","faster","reduce time","time to green","parallel","shard","concurr"]) and \
       any(c in s for c in [" ci","build","pipeline","workflow","runner","gha","github actions","gitlab","jenkins","circleci","azure pipelines","bitrise"]):
        hits.add("speed_up_ci")
    if any(w in s for w in ["gradle","agp","jdk","java","version","wrapper","target api"]):
        hits.add("version_change")
    if any(w in s for w in ["migrate","switch","replace","port","github actions","gha","gitlab","jenkins","circleci","azure pipelines","bitrise","workflow","pipeline"]):
        hits.add("ci_env_workflow")
    hits = sorted(hits)
    reason = hits[0] if hits else None
    return hits, reason

# -----------------------------
# 7-subintent + 4-headline mapping
# -----------------------------
RAW_TO_SUBINTENT = {
    "expand_coverage":"coverage", "reduce_coverage":"coverage", "coverage":"coverage",
    "flake_mitigation":"flake_mitigation", "speed_up_ci":"runtime",
    "version_change":"version_change",
    "ci_env_workflow":"ci_platform_infra",
}
SUB_TO_HEADLINE = {
    "coverage":"coverage",
    "flake_mitigation":"runtime",
    "runtime":"runtime",
    "version_change":"version_change",
    "ci_platform_infra":"ci_platform_infra",
}
SUB_PRIORITY = ["coverage","flake_mitigation","runtime","version_change","ci_platform_infra"]

def to_subbuckets(raws: List[str]) -> List[str]:
    out = []
    for r in raws or []:
        b = RAW_TO_SUBINTENT.get(r)
        if b and b not in out:
            out.append(b)
    return out

def pick_by_priority(cands: List[str]) -> str:
    for b in SUB_PRIORITY:
        if b in cands:
            return b
    return ""

# -----------------------------
# Style intent (optional): map style transitions to raw intents
# -----------------------------
def style_to_intent(style_old: Optional[str], style_new: Optional[str]) -> List[str]:
    out = []
    if not style_old or not style_new or style_old == style_new:
        return out
    if style_new == "gmd":
        out.append("adopt_gmd")
    elif style_new == "diy":
        out.append("adopt_diy")
    else:
        out.append("genericize_emulator")
    return out

# -----------------------------
# Change-op utils (robust)
# -----------------------------
def classify_change_op(old_value: str, new_value: str, change_type: str) -> str:
    ct = (change_type or "").lower()
    if ct == "added":   return "add"
    if ct == "removed": return "remove"

    # Fallback: infer from payloads if possible
    try:
        o = json.loads(old_value) if old_value else []
    except: o = []
    try:
        n = json.loads(new_value) if new_value else []
    except: n = []

    if (not o) and n:
        return "add"
    if o and (not n):
        return "remove"
    return "value_edit" if (old_value or "") != (new_value or "") else "no_change"

# -----------------------------
# Step-1 labels (no feature_scope; we only consume path-kind overlaps)
# -----------------------------
def _get_labels(feats: dict) -> dict:
    return (feats or {}).get("labels", {}) or {}

def compute_field_group(primary_category: str) -> str:
    if primary_category in {CAT_COVERAGE, CAT_RUNTIME}:
        return "emulator_related"
    return "context_only"

def spot_tag_for_filekind(file_kind: str) -> Optional[str]:
    if file_kind == "ci_yaml": return "yaml_unscoped"
    if file_kind == "gradle":  return "gradle_global"
    return None

# -----------------------------
# Main
# -----------------------------
if __name__ == "__main__":
    print(f"[info] snapshots dir: {SNAPSHOT_DIR}")
    print(f"[info] output dir   : {OUT_DIR}")

    by_repo = read_snapshots(SNAPSHOT_DIR)
    print(f"[info] Loaded snapshots for {len(by_repo)} repos")

    for repo, rows in by_repo.items():
        out_rows: List[dict] = []
        by_path: Dict[str, List[dict]] = {}
        for r in rows:
            by_path.setdefault(r.get("path",""), []).append(r)

        repeat_counter: Dict[Tuple[str,str], int] = {}

        for path, snaps in by_path.items():
            prev: Optional[dict] = None
            for cur in snaps:
                # --- NO BASELINE EMISSION ---
                if prev is None:
                    prev = cur
                    continue

                old_feats = prev.get("features", {}) or {}
                new_feats = cur.get("features", {}) or {}

                deltas, episode_aux = diff_features(old_feats, new_feats)
                if deltas:
                    # Episode delta intents
                    intent_delta_episode = derive_delta_intents(deltas)

                    subject_str = cur.get("subject","")
                    ts_epoch_utc = ensure_utc_epoch(cur.get("timestamp", 0))
                    ts_iso_utc   = epoch_to_iso_utc(ts_epoch_utc)

                    # Step-1 labels (path/file context only)
                    new_labels = _get_labels(new_feats)
                    file_kind = new_labels.get("file_kind", "")
                    ci_emu_change = new_labels.get("ci_emulator_step_change", None)
                    gmd_block_change = new_labels.get("gradle_gmd_block_change", None)

                    # Episode-level: style & GMD/Orchestrator direction
                    style_old = episode_aux.get("Emulator_Style_Old")
                    style_new = episode_aux.get("Emulator_Style_New")
                    style_change = f"{style_old.upper()}→{style_new.upper()}" if (style_old and style_new and style_old != style_new) else ""

                    gmd_old = episode_aux.get("GMD_Present_Old")
                    gmd_new = episode_aux.get("GMD_Present_New")
                    if gmd_old is True and gmd_new is False:
                        gmd_dir = "GMD→DIY"
                    elif gmd_old is False and gmd_new is True:
                        gmd_dir = "DIY→GMD"
                    else:
                        gmd_dir = ""

                    orch_old = episode_aux.get("Orchestrator_Enabled_Old")
                    orch_new = episode_aux.get("Orchestrator_Enabled_New")
                    if orch_old is True and orch_new is False:
                        orch_change = "disable"
                    elif orch_old is False and orch_new is True:
                        orch_change = "enable"
                    else:
                        orch_change = ""

                    # Intent helpers (for reason text only)
                    helper_tags = []
                    got_helpers = False
                    for dp in (new_feats.get("detected_params") or []):
                        if dp.get("context_anchor"): helper_tags.append(f"ctx={dp['context_anchor']}")
                        if dp.get("test_command"):   helper_tags.append("has_test_cmd")
                        if dp.get("runner_args"):    helper_tags.append("has_runner_args")
                        got_helpers = True
                        break
                    intent_helper_note = ";".join(helper_tags) if got_helpers else None

                    for d in deltas:
                        field = d["field"]
                        key = (path, field)
                        repeat_counter[key] = repeat_counter.get(key, 0) + 1
                        repeat_index = repeat_counter[key]
                        repeat_label = "first" if repeat_index == 1 else "repeat"

                        # WHAT + category
                        primary_label = FIELD_TO_PRIMARY.get(field, "other_change")
                        primary_category = CATEGORY_NAME.get(
                            PRIMARY_TO_CATEGORY.get(primary_label, CAT_CONTEXT),
                            PRIMARY_TO_CATEGORY.get(primary_label, CAT_CONTEXT)
                        )

                        # Per-source intents
                        intents_delta_list = derive_delta_intents([d]) if field in DELTA_MEANINGFUL_FIELDS else []
                        intents_path = detect_intent_path_generic(path)
                        intents_pfa_list, intent_pfa_reason = detect_intent_path_field_aware(path, field)
                        intents_subject, subj_reason = detect_subject_intents(subject_str)
                        style_intents = style_to_intent(style_old, style_new)

                        # Attach helper note to path reason (no schema change)
                        if intent_helper_note:
                            intent_pfa_reason = f"{intent_pfa_reason} | {intent_helper_note}" if intent_pfa_reason else intent_helper_note

                        # Arbitration
                        source_to_pool = [
                            ("Delta",   intents_delta_list,  "delta-derived"),
                            ("Style",   style_intents,       "emulator style transition"),
                            ("PathFA",  intents_pfa_list,    intent_pfa_reason or "path field-aware"),
                            ("Subject", intents_subject,     subj_reason or "subject regex"),
                            ("Path",    intents_path,        "path generic"),
                        ]
                        chosen_source = ""; chosen_raws: List[str] = []; chosen_reason = ""
                        for s, pool, why in source_to_pool:
                            if pool:
                                chosen_source, chosen_raws, chosen_reason = s, sorted(set(pool)), why
                                break

                        secondary_label = ",".join(chosen_raws) if chosen_raws else ""
                        sub_cands = to_subbuckets(chosen_raws)
                        driver = pick_by_priority(sub_cands) if sub_cands else ""
                        headline = SUB_TO_HEADLINE.get(driver, "") if driver else ""

                        change_op = classify_change_op(d.get("old_value",""), d.get("new_value",""), d.get("change_type","modified"))
                        field_group = compute_field_group(primary_category)
                        spot_tag = spot_tag_for_filekind(file_kind)
                        include_emu_study = (field_group == "emulator_related")

                        # --- Output row
                        row = {
                            # Metadata
                            "repo": repo,
                            "sha": cur.get("sha"),
                            "prev_sha": prev.get("sha"),
                            "timestamp_epoch_utc": ts_epoch_utc,
                            "timestamp_utc": ts_iso_utc,
                            "path": path,
                            "subject": subject_str,

                            # Primary (WHAT)
                            "Field": field,
                            "Primary_Label": primary_label,
                            "Primary_Label_Category": primary_category,

                            # Diff details
                            "old_value": d.get("old_value",""),
                            "new_value": d.get("new_value",""),
                            "change_type": d.get("change_type","modified"),
                            "change_op": change_op,
                            "magnitude": d.get("magnitude", None),
                            "added_items": d.get("added_items",""),
                            "removed_items": d.get("removed_items",""),
                            "repeat_index": repeat_index,
                            "repeat_label": repeat_label,

                            # Per-source intents
                            "intent_delta": ",".join(sorted(set(intents_delta_list))),
                            "intent_pfa": ",".join(sorted(set(intents_pfa_list))),
                            "intent_subject": ",".join(sorted(set(intents_subject))),
                            "intent_path": ",".join(sorted(set(intents_path))),
                            "intent_delta_episode": ",".join(sorted(set(intent_delta_episode))) if intent_delta_episode else "",

                            # Secondary (chosen raw intents)
                            "Secondary_Label": secondary_label,

                            # Why (sub-bucket/headline)
                            "Driver": driver,
                            "intent_headline": headline,
                            "Driver_Source": chosen_source or "",
                            "Driver_reason": chosen_reason or "",

                            # Aux (schema-compat + reasons)
                            "Aux_Tags": "",
                            "Intent_Path": "; ".join(intents_pfa_list) if intents_pfa_list else None,
                            "Intent_Path_Reason": intent_pfa_reason,
                            "Intent_Subject_Reason": subj_reason,

                            # Legacy compat
                            "Driver_Label_Selected": driver,

                            # Step-1 carryover
                            "File_Kind": file_kind,                          # ci_yaml|gradle|script|other
                            "CI_Emulator_Step_Change": ci_emu_change,        # True/False/None
                            "Gradle_GMD_Block_Change": gmd_block_change,     # True/False/None
                            "Context_Scope_Keys": "",                        # none in new miner
                            "Is_Emulator_Relevant_File": bool(ci_emu_change or gmd_block_change),

                            # Grouping
                            "Field_Group": field_group,                       # emulator_related|context_only
                            "Spot_Tag": spot_tag,
                            "Include_Emulator_Study": bool(include_emu_study),
                        }

                        # --- AUX: style & GMD/Orchestrator on every row
                        row["Emulator_Style_Old"] = style_old or ""
                        row["Emulator_Style_New"] = style_new or ""
                        row["Emulator_Style_Change"] = style_change

                        row["GMD_Present_Old"] = gmd_old
                        row["GMD_Present_New"] = gmd_new
                        row["GMD_Direction"]   = gmd_dir

                        row["Orchestrator_Enabled_Old"] = orch_old
                        row["Orchestrator_Enabled_New"] = orch_new
                        row["Orchestrator_Change"]      = orch_change

                        # (Per-delta extras if present): Coverage_Added_Count, Coverage_Removed_Count, Coverage_Net
                        # Runtime_Wait_Delta, Runtime_Wait_Change (for boot_wait_seconds)

                        out_rows.append(row)

                prev = cur  # advance window

        if not out_rows:
            print(f"[skip] {repo}: no field-level deltas found")
            continue

        dst = OUT_DIR / f"{repo}.jsonl"
        with dst.open("w", encoding="utf-8") as f:
            for r in out_rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"[ok] {repo}: {len(out_rows)} enriched rows -> {dst}")

    print("Done.")


[info] snapshots dir: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshotsV8.1
[info] output dir   : C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enrichedV8.1
[info] Loaded snapshots for 9 repos
[skip] Arian04__android-hid-client: no field-level deltas found
[skip] arkivanov__android-dev-challenge-compose-2: no field-level deltas found
[skip] elftausend__custos: no field-level deltas found
[skip] guolindev__android-dev-challenge-compose: no field-level deltas found
[skip] ICBNetwork__web3-Wallet: no field-level deltas found
[skip] okanaydin__Android-Compose-Challenge: no field-level deltas found
[skip] prisma__react-native-prisma: no field-level deltas found
[skip] riggaroo__AndroidDatabaseUpgrades: no field-level deltas found
[skip] tfcporciuncula__phonemoji: no field-level deltas found
Done.


## Step 4: Combine

In [ ]:
# combine_v81.py
# RQ2 — Step 4 (Combine v8.1): Snapshots + Enriched Episodes
# - Aligned with Step-2 v8.1 (NO baseline "begin")
# - Column order: Metadata → Other → Primary → Secondary → Driver (+ AUX)
# - Cleans list-like cells to CSV-friendly strings
# - PRESERVES JSON diff payloads (old_value/new_value/added_items/removed_items) verbatim
# - NO Parquet output

from __future__ import annotations
import json, csv, ast, math
from pathlib import Path
from typing import List, Any

# -----------------------------
# Config (aligned with v7.0 paths)
# -----------------------------
WORK_ROOT         = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
SNAPSHOT_DIR      = WORK_ROOT / "snapshotsV8.1"
CCE_ENRICHED_DIR  = WORK_ROOT / "cce_enrichedV8.1"
COMBINE_DIR       = WORK_ROOT / "combinedV8.1"
COMBINE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helpers
# -----------------------------
def read_all_jsonl(folder: Path) -> List[dict]:
    out: List[dict] = []
    if not folder.exists():
        return out
    for p in folder.glob("*.jsonl"):
        with p.open(encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    d = json.loads(line)
                    d["_source_file"] = p.name
                    out.append(d)
    return out

def to_json(x: Any) -> str:
    try:
        return json.dumps(x, ensure_ascii=False, sort_keys=True)
    except Exception:
        return "" if x is None else str(x)

def _is_na_like(x: Any) -> bool:
    if x is None:
        return True
    if isinstance(x, float):
        try:
            return math.isnan(x)
        except Exception:
            return False
    if isinstance(x, str):
        s = x.strip().lower()
        return s in {"", "null", "none", "nan", "na"}
    return False

def conservative_change_op(old_v: Any, new_v: Any) -> str:
    # Fallback only; Step-2 already emits 'change_op'
    o_empty = _is_na_like(old_v)
    n_empty = _is_na_like(new_v)
    if o_empty and n_empty:
        return "no_change"
    if o_empty and not n_empty:
        return "add"
    if not o_empty and n_empty:
        return "remove"
    return "value_edit" if (str(old_v) != str(new_v)) else "no_change"

def to_list_like(x: Any) -> list:
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, (set, tuple)):
        return list(x)
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        # JSON- or Python-like list
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                v = json.loads(s)
                if isinstance(v, list):
                    return v
            except Exception:
                pass
            try:
                v = ast.literal_eval(s)
                if isinstance(v, (list, tuple, set)):
                    return list(v)
            except Exception:
                pass
        # Delimited
        if "," in s:
            return [t.strip() for t in s.split(",") if t.strip()]
        if ";" in s:
            return [t.strip() for t in s.split(";") if t.strip()]
        # Single token
        return [s]
    # Fallback
    return [x]

# Keep these exact JSON strings; don't normalize them to CSV tokens
KEEP_AS_JSON = {"old_value", "new_value", "added_items", "removed_items"}

def clean_cell_no_brackets(v: Any) -> str:
    """Default cleaner: lists→'a,b,c', dicts→JSON, strip brackets for stringified lists."""
    if v is None:
        return ""
    if isinstance(v, bool):
        return "True" if v else "False"
    if isinstance(v, dict):
        try:
            return json.dumps(v, ensure_ascii=False, sort_keys=True)
        except Exception:
            return str(v)
    if isinstance(v, (list, tuple, set)) or (isinstance(v, str) and v.strip()[:1] in "[(" and v.strip()[-1:] in "])"):
        items = [str(t).strip() for t in to_list_like(v) if str(t).strip()]
        items = sorted(set(items))
        return ",".join(items)
    s = str(v).strip()
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        items = [str(t).strip() for t in to_list_like(s) if str(t).strip()]
        items = sorted(set(items))
        return ",".join(items)
    return str(v)

def clean_cell_preserve_json(key: str, v: Any) -> str:
    """Preserve JSON strings for diff payloads; otherwise use the normal cleaner."""
    if key in KEEP_AS_JSON and isinstance(v, str):
        return v.strip()
    return clean_cell_no_brackets(v)

# -----------------------------
# Load inputs
# -----------------------------
snapshots = read_all_jsonl(SNAPSHOT_DIR)
episodes  = read_all_jsonl(CCE_ENRICHED_DIR)
print(f"Loaded {len(snapshots)} snapshots; {len(episodes)} enriched episode rows.")

# -----------------------------
# Write CSV (snapshots)
# -----------------------------
snap_csv = COMBINE_DIR / "snapshots_combined.csv"
if snapshots:
    snaps_flat = []
    for r in snapshots:
        feats = r.get("features", {})
        rr = {**r, "features_json": to_json(feats)}
        rr.pop("features", None)
        snaps_flat.append(rr)

    keys = sorted(set().union(*[set(x.keys()) for x in snaps_flat]))
    with snap_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in snaps_flat:
            r_clean = {k: clean_cell_no_brackets(v) for k, v in r.items()}
            w.writerow(r_clean)
    print(f"[ok] {snap_csv}")
else:
    print("[warn] No snapshots found.")

# -----------------------------
# Write CSV (episodes enriched) — align to Step-2 v7.0 (no baseline)
# -----------------------------
cce_csv = COMBINE_DIR / "episodes_enriched_combined.csv"
if episodes:
    # Groups → Metadata / Other / Primary / Secondary / Driver (+ AUX)
    META_COLS = [
        "repo", "_source_file", "sha", "prev_sha",
        "timestamp_epoch_utc", "timestamp_utc",
        "path", "subject",
    ]

    # Diff/audit (keep JSON payloads)
    DIFF_COLS = [
        "old_value", "new_value", "change_type", "change_op", "magnitude",
        "added_items", "removed_items", "repeat_index", "repeat_label",
        # Per-delta extras (may be blank unless applicable)
        "Coverage_Added_Count", "Coverage_Removed_Count", "Coverage_Net",
        "Runtime_Wait_Delta", "Runtime_Wait_Change",
        "Aux_Tags",
        "intent_delta_episode",
    ]

    # Step-1 carryover (minimal)
    STEP1_CARRY_COLS = [
        "File_Kind",
        "CI_Emulator_Step_Change",
        "Gradle_GMD_Block_Change",
        "Context_Scope_Keys",
        "Is_Emulator_Relevant_File",
    ]

    # Grouping & flags
    GROUPING_AND_FLAGS = [
        "Spot_Tag",
        "Field_Group",
        "Include_Emulator_Study",
    ]

    # AUX fields emitted by Step-2 v7.0
    AUX_COLS = [
        # Emulator style (episode-level)
        "Emulator_Style_Old", "Emulator_Style_New", "Emulator_Style_Change",
        # GMD presence (+ direction)
        "GMD_Present_Old", "GMD_Present_New", "GMD_Direction",
        # Orchestrator presence (+ change)
        "Orchestrator_Enabled_Old", "Orchestrator_Enabled_New", "Orchestrator_Change",
    ]

    OTHER_COLS = DIFF_COLS + STEP1_CARRY_COLS + GROUPING_AND_FLAGS + AUX_COLS

    PRIMARY_COLS = [
        "Field",
        "Primary_Label",
        "Primary_Label_Category",
    ]

    SECONDARY_COLS = [
        "intent_delta", "intent_pfa", "intent_subject", "intent_path",
        "Secondary_Label",
        "Intent_Path", "Intent_Path_Reason", "Intent_Subject_Reason",
    ]

    DRIVER_COLS = [
        "Driver",
        "intent_headline",
        "Driver_Source",
        "Driver_reason",
        "Driver_Label_Selected",
    ]

    ORDERED_COLS = META_COLS + OTHER_COLS + PRIMARY_COLS + SECONDARY_COLS + DRIVER_COLS

    # Build rows (only fields from known schema; unknown keys ignored)
    rows_out: List[dict] = []
    for r in episodes:
        rr = {}
        for k in ORDERED_COLS:
            if k in r:
                rr[k] = r.get(k, "")
        # conservative fill for change_op if missing (fallback only)
        if not rr.get("change_op"):
            rr["change_op"] = conservative_change_op(r.get("old_value"), r.get("new_value"))
        # Clean cells (preserve JSON diff payloads)
        rr = {k: clean_cell_preserve_json(k, v) for k, v in rr.items()}
        rows_out.append(rr)

    # Header: include only columns that appear at least once, but force core metadata
    present_cols = [c for c in ORDERED_COLS if any((c in r and r[c] != "") for r in rows_out)]
    for c in META_COLS:
        if c not in present_cols:
            present_cols.insert(0 if c == "repo" else len(present_cols), c)

    with cce_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=present_cols)
        w.writeheader()
        for r in rows_out:
            w.writerow({k: r.get(k, "") for k in present_cols})

    print(f"[ok] {cce_csv}")
else:
    print("[warn] No enriched episodes found.")


Loaded 278584 snapshots; 14529 enriched episode rows.
[ok] C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combinedV8.1\snapshots_combined.csv
[ok] C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combinedV8.1\episodes_enriched_combined.csv
